# Phase 7 — BRID controlled hybrid matching

## Goal

Run the approved synthetic BRID case through the hardened Phase 5.2 engine,
assemble a portfolio that respects count-based eligibility rules, and create a
citation-audit workbook.

Every output is `SYNTHETIC_TEST_ONLY`. This notebook never awards tender points,
approves a business shortlist, or enables production promotion.

## Run Phase 7

Run the single cell below. It is deliberately self-contained: a fresh Colab
kernel cannot inherit stale imports or outputs from an earlier session. The
first clean run may take several minutes while the pinned multilingual E5 model
downloads; embeddings execute locally.

In [ ]:
from pathlib import Path
import hashlib
import importlib
import importlib.util
import json
import os
import subprocess
import sys
import zipfile

PROJECT_PARENT_FOLDER_NAME = "Devoteam internship"
PROJECT_FOLDER_NAME = "Devoteam_AI_CLEAN_PIPELINE"
PACKAGE_FILENAME = "PHASE_7_BRID_HYBRID_MATCHING_PACKAGE.zip"
PACKAGE_SHA256 = "e61e76aeea5df4581248d6d0d6f505361dfd756cf4f278d269cd4b01b5bd2966"
PACKAGE_MANIFEST_SHA256 = "151e7edfd3fb92fb59ad3b115429d3ca5225786298c6cffa1ae487840ff3224e"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def activate_project_source(project_root):
    source_path = (Path(project_root) / "src").resolve()
    package_init = source_path / "devoteam_reference_ai" / "__init__.py"
    assert package_init.is_file(), f"Missing project package: {package_init}"
    source_root = str(source_path)
    sys.path[:] = [entry for entry in sys.path if entry != source_root]
    sys.path.insert(0, source_root)
    inherited = [
        entry
        for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep)
        if entry and entry != source_root
    ]
    os.environ["PYTHONPATH"] = os.pathsep.join([source_root, *inherited])
    importlib.invalidate_caches()
    spec = importlib.util.find_spec("devoteam_reference_ai")
    assert spec is not None and spec.origin is not None, (
        "The project package is not importable after activation"
    )
    assert Path(spec.origin).resolve() == package_init.resolve(), (
        f"Wrong project package resolved: {spec.origin}"
    )
    return source_root


print("1/5 — Resolving the verified project")
override = os.environ.get("DEVOTEAM_PROJECT_ROOT", "").strip()
if override:
    PROJECT_ROOT = Path(override).resolve()
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = (
        Path("/content/drive/MyDrive")
        / PROJECT_PARENT_FOLDER_NAME
        / PROJECT_FOLDER_NAME
    ).resolve()

required_project_files = [
    PROJECT_ROOT / "README_START_HERE.md",
    PROJECT_ROOT / "config" / "phase5_retrieval.yaml",
    PROJECT_ROOT / "config" / "phase5_2_matching_hardening.yaml",
    PROJECT_ROOT / "data" / "opportunities" / "OPP-2098dd1874292d22"
        / "phase6_brid_controlled_case_v2" / "BRID_PHASE_6_REVIEW.xlsx",
]
missing = [str(path) for path in required_project_files if not path.is_file()]
assert not missing, "Required project input is missing:\n" + "\n".join(missing)
assert PROJECT_ROOT.name == PROJECT_FOLDER_NAME, PROJECT_ROOT
PACKAGE_PATH = PROJECT_ROOT / PACKAGE_FILENAME
assert PACKAGE_PATH.is_file(), f"Missing Phase 7 package: {PACKAGE_PATH}"
print(f"PROJECT_ROOT={PROJECT_ROOT}")

print("2/5 — Verifying the signed Phase 7 overlay")
assert file_sha256(PACKAGE_PATH) == PACKAGE_SHA256, "Phase 7 package hash mismatch"
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    names = archive.namelist()
    assert "PHASE_7_BRID_PACKAGE_MANIFEST.json" in names
    manifest_bytes = archive.read("PHASE_7_BRID_PACKAGE_MANIFEST.json")
    assert hashlib.sha256(manifest_bytes).hexdigest() == PACKAGE_MANIFEST_SHA256
    package_manifest = json.loads(manifest_bytes)
    allowed = set(package_manifest["files"]) | {"PHASE_7_BRID_PACKAGE_MANIFEST.json"}
    assert set(names) == allowed, "Package contains undeclared files"
    installed = skipped = 0
    for name in names:
        target = (PROJECT_ROOT / name).resolve()
        assert PROJECT_ROOT in target.parents, name
        data = archive.read(name)
        if name != "PHASE_7_BRID_PACKAGE_MANIFEST.json":
            assert hashlib.sha256(data).hexdigest() == package_manifest["files"][name]["sha256"]
        if target.exists():
            assert target.read_bytes() == data, f"Conflicting Phase 7 BRID file: {name}"
            skipped += 1
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            installed += 1
print(f"Overlay verified: installed={installed}, identical={skipped}")

print("3/5 — Installing dependencies and running 45 controls")
if os.environ.get("DEVOTEAM_SKIP_INSTALL", "") != "1":
    install_result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(PROJECT_ROOT / "requirements" / "phase7_brid.txt"),
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if install_result.returncode != 0:
        raise RuntimeError(
            "Dependency installation failed.\n" + install_result.stdout[-12000:]
        )

source_root = activate_project_source(PROJECT_ROOT)
tests = [
    PROJECT_ROOT / "tests" / "test_phase5_2_matching.py",
    PROJECT_ROOT / "tests" / "test_phase6_brid_contract.py",
    PROJECT_ROOT / "tests" / "test_phase7_brid_matching.py",
]
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = os.pathsep.join(
    value
    for value in [source_root, test_environment.get("PYTHONPATH", "")]
    if value
)
test_environment["PYTHONUNBUFFERED"] = "1"
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", *map(str, tests)],
    cwd=PROJECT_ROOT,
    env=test_environment,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(test_result.stdout, end="")
if test_result.returncode != 0:
    raise RuntimeError(
        "Focused regression controls failed "
        f"(pytest exit code {test_result.returncode}).\n"
        + test_result.stdout[-12000:]
    )
print("Phase 5.2–7 focused regression controls: PASS")

print("4/5 — Verifying the approved workbook and signed inputs")
activate_project_source(PROJECT_ROOT)
from devoteam_reference_ai.phase7_brid_matching import (
    PinnedE5QueryAdapter,
    load_approved_brid_review,
    load_phase7_brid_config,
    run_phase7_brid,
    verify_phase5_2_inputs,
    verify_phase7_brid_run,
)

PHASE7_CONFIG_PATH = PROJECT_ROOT / "config" / "phase7_brid_controlled_case.yaml"
PHASE7_CONFIG = load_phase7_brid_config(PHASE7_CONFIG_PATH)
PHASE6_ROOT = (
    PROJECT_ROOT / "data" / "opportunities"
    / PHASE7_CONFIG["input"]["opportunity_id"]
    / PHASE7_CONFIG["input"]["phase6_run_name"]
)
REVIEW = load_approved_brid_review(
    PHASE6_ROOT / PHASE7_CONFIG["input"]["review_workbook_name"],
    PHASE7_CONFIG,
)
SIGNED_INPUTS = verify_phase5_2_inputs(PROJECT_ROOT, PHASE7_CONFIG)
print(
    "Approved controls:",
    len(REVIEW["match_requirements"]), "match requirements,",
    len(REVIEW["policy_requirements"]), "policy requirements,",
    len(REVIEW["eligibility_rules"]), "eligibility rules,",
    len(REVIEW["scoring_criteria"]), "scoring criteria,",
    len(REVIEW["user_facets"]), "non-automatic user facets.",
)

print("5/5 — Running pinned local E5 hybrid matching")
EMBEDDING_ADAPTER = PinnedE5QueryAdapter(
    PROJECT_ROOT / "config" / "phase5_retrieval.yaml"
)
RUN_ROOT, MANIFEST = run_phase7_brid(
    project_root=PROJECT_ROOT,
    config_path=PHASE7_CONFIG_PATH,
    embedding_adapter=EMBEDDING_ADAPTER,
)
VERIFICATION = verify_phase7_brid_run(RUN_ROOT, PHASE7_CONFIG)
assert VERIFICATION["manifest"] == MANIFEST
assert MANIFEST["status"] in {
    "TECHNICAL_PASS_READY_FOR_EVIDENCE_AUDIT",
    "TECHNICAL_PASS_WITH_PORTFOLIO_GAPS",
}

print("PHASE 7 BRID HYBRID MATCHING: PASS")
print(f"Status: {MANIFEST['status']}")
print(f"Authorized signed pool: {MANIFEST['authorized_signed_reference_pool']}")
print(f"Candidate references: {MANIFEST['candidate_references']}")
print(f"Portfolio references: {MANIFEST['portfolio_references']}")
print(
    "Portfolio rules:",
    f"{MANIFEST['portfolio_rules_passed']}/{MANIFEST['portfolio_rules_total']}",
)
print(
    "MUST requirement union coverage:",
    f"{MANIFEST['must_requirements_covered']}/{MANIFEST['must_requirements_total']}",
)
print(f"Citation completeness: {MANIFEST['citation_completeness']:.0%}")
print(f"Citation integrity: {MANIFEST['citation_integrity']:.0%}")
print(f"Citation correctness: {MANIFEST['citation_correctness_status']}")
print(f"Tender threshold: {MANIFEST['technical_threshold_status']}")
print(f"Output: {RUN_ROOT}")

## Next step

Open `BRID_PHASE_7_REVIEW.xlsx` in the printed output folder. Audit every cited
passage before marking a portfolio row `SHORTLIST` or `REJECT`.

Do not enter or claim a tender score here. Team and methodology remain pending,
citation correctness requires human audit, and Phase 5.1 still blocks production
promotion.